In [0]:
# Cell 1 - Create a larger benchmark dataset
# Load existing order_items
order_items = spark.table("shopsphere_catalog.retail.order_items")
orders = spark.table("shopsphere_catalog.retail.orders")
products = spark.table("shopsphere_catalog.retail.products")

print(f"Base order_items count: {order_items.count():,}")
# Create 50x larger dataset for meaningful benchmarks
from pyspark.sql.functions import col, lit, rand, expr
# Repeat the dataset with slight variation to avoid caching effects
order_items_large = order_items
for i in range(1, 50):
    variation = order_items.withColumn("price", col("price") * (1 + rand() * 0.1))
    order_items_large = order_items_large.union(variation)

order_items_large = order_items_large.repartition(50)
print(f"Benchmark dataset size: {order_items_large.count():,} rows")

In [0]:
# Step 2:  Run Benchmark Without Photon
# Benchmark WITHOUT Photon
import time
from pyspark.sql.functions import sum as _sum, avg, count, round as _round, col, date_format

# Start timer
start = time.time()

# Benchmark Query: Complex aggregation across joined tables
result_no_photon = (
    order_items_large
    .join(orders.select("order_id", "order_status", "order_purchase_timestamp"), "order_id")
    .join(products.select("product_id", "product_category_name"), "product_id")
    .filter(col("order_status") == "DELIVERED")
    .withColumn("order_month", date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("product_category_name", "order_month")
    .agg(
        _round(_sum("price"), 2).alias("total_revenue"),
        _round(avg("price"), 2).alias("avg_price"),
        count("order_id").alias("order_count"),
        _round(_sum("freight_value"), 2).alias("total_freight"))
    .filter(col("total_revenue") > 1000)
    .orderBy("total_revenue", ascending=False))
result_no_photon.count()  # Force full execution
elapsed_no_photon = time.time() - start
print(f"\n WITHOUT Photon: {elapsed_no_photon:.2f} seconds")
print(f"Rows returned: {result_no_photon.count()}")


In [0]:
# Step 3:  Run Benchmark With Photon

# Benchmark WITH Photon
import time
from pyspark.sql.functions import sum as _sum, avg, count, round as _round, col, date_format
# Re-create the cache on Photon cluster (re-run Step 1 cache setup)
# order_items_large.cache() then .count()
start = time.time()
# EXACT same query as before
result_with_photon = (
    order_items_large
    .join(orders.select("order_id", "order_status", "order_purchase_timestamp"), "order_id")
    .join(products.select("product_id", "product_category_name"), "product_id")
    .filter(col("order_status") == "DELIVERED")
    .withColumn("order_month", date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("product_category_name", "order_month")
    .agg(
        _round(_sum("price"), 2).alias("total_revenue"),
        _round(avg("price"), 2).alias("avg_price"),
        count("order_id").alias("order_count"),
        _round(_sum("freight_value"), 2).alias("total_freight"))
    .filter(col("total_revenue") > 1000)
    .orderBy("total_revenue", ascending=False))
result_with_photon.count()
elapsed_with_photon = time.time() - start
print(f"\n WITH Photon: {elapsed_with_photon:.2f} seconds")
print(f"Speedup factor: {elapsed_no_photon / elapsed_with_photon:.1f}x faster")


In [0]:
# Step 4:  Test Photon with MERGE (Most Impactful Use Case)
# Create a staging table to simulate incremental orders
from pyspark.sql.functions import current_timestamp, lit

# Create target table (order summary)
spark.sql("""
    CREATE OR REPLACE TABLE shopsphere_catalog.retail.orders_benchmark_target
    USING DELTA AS
    SELECT order_id, order_status, 0 AS processing_version
    FROM shopsphere_catalog.retail.orders
    LIMIT 50000
""")
# Create source (new/updated records)
source_updates = spark.table("shopsphere_catalog.retail.orders") \
    .limit(10000) \
    .withColumn("order_status", lit("PROCESSED"))
source_updates.createOrReplaceTempView("updates_source")
# Benchmark MERGE operation
import time
start_merge = time.time()
spark.sql("""
MERGE INTO shopsphere_catalog.retail.orders_benchmark_target AS target
USING updates_source AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
  target.order_status = source.order_status,
  target.processing_version = target.processing_version + 1
WHEN NOT MATCHED THEN INSERT (
  order_id,
  order_status,
  processing_version)VALUES (
  source.order_id,
  source.order_status, 1)
""")
merge_time = time.time() - start_merge
print(f"MERGE elapsed: {merge_time:.2f} seconds")
print("Photon provides 3-8x speedup on MERGE operations vs standard engine")


In [0]:
# Cell 1 - Setup streaming infrastructure
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
import random
# Define the streaming source table (event log)
EVENT_SOURCE_TABLE = "shopsphere_catalog.retail.order_events_stream"
CHECKPOINT_BASE = "/tmp/shopsphere/checkpoints"
# Create the event source table (Delta table as streaming source)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {EVENT_SOURCE_TABLE} (
        event_id        STRING,
        order_id        STRING,
        customer_id     STRING,
        event_type      STRING,    -- 'ORDER_PLACED', 'PAYMENT_CONFIRMED', 'CANCELLED'
        product_id      STRING,
        order_value_brl DOUBLE,
        seller_state    STRING,
        customer_state  STRING,
        event_timestamp TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true')
""")
print(f" Streaming source table created: {EVENT_SOURCE_TABLE}")


In [0]:
# Cell 2 — Event generator function

import uuid
import random
import builtins
from datetime import datetime

def generate_order_events(num_events=100, batch_id=0):
    """Simulate order events arriving from ShopSphere app."""
    
    states = ['SP', 'RJ', 'MG', 'BA', 'PR', 'RS', 'PE', 'CE', 'GO', 'MA']
    event_types = ['ORDER_PLACED'] * 7 + ['PAYMENT_CONFIRMED'] * 2 + ['CANCELLED'] * 1
    events = []
    for i in range(num_events):
        order_value = builtins.round(random.uniform(20, 12000), 2)
        events.append({
            'event_id': str(uuid.uuid4()),
            'order_id': f"ORD-{batch_id}-{i:04d}",
            'customer_id': f"CUST-{random.randint(1000, 99999)}",
            'event_type': random.choice(event_types),
            'product_id': f"PROD-{random.randint(100, 9999)}",
            'order_value': order_value,
            'seller_state': random.choice(states),
            'customer_state': random.choice(states),
            'event_timestamp': datetime.now()
        })
    
    return events


In [0]:
# Test the generator
sample_events = generate_order_events(5, 0)
for e in sample_events:
    print(f"{e['event_id'][:8]}... | {e['event_type']} | BRL {e['order_value']}")


In [0]:
# Step 4:  Ingest Initial Event Batch
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col
# Define schema (same as generator)
schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_value", DoubleType(), False),
    StructField("seller_state", StringType(), False),
    StructField("customer_state", StringType(), False),
    StructField("event_timestamp", TimestampType(), False)])
# Generate events
batch_0_events = generate_order_events(1000, batch_id=0)
# Create DataFrame
batch_0_df = spark.createDataFrame(batch_0_events, schema=schema)
batch_0_df = batch_0_df.withColumnRenamed("order_value", "order_value_brl")
# Append safely (NO overwrite, NO drop)
batch_0_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(EVENT_SOURCE_TABLE)
# Verify
print(f"Batch 0 written: {batch_0_df.count()} events")
print(f"Total events in source: {spark.table(EVENT_SOURCE_TABLE).count()}")


In [0]:
order_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")       # Handle updates to source table
    .option("maxFilesPerTrigger", "10")    # Control batch size
    .table(EVENT_SOURCE_TABLE))
print("Stream reader created")
print("Schema:")
order_stream.printSchema()


In [0]:
# STEP B: Transform - Add business logic
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, 
    window, sum as _sum, count, round as _round)
# Transform 1: Raw events with enrichment
order_stream_enriched = order_stream \
    .withColumn("ingest_timestamp", current_timestamp()) \
    .withColumn("order_size_category", 
        when(col("order_value_brl") < 100, "micro")
        .when(col("order_value_brl") < 500, "small")
        .when(col("order_value_brl") < 2000, "medium")
        .when(col("order_value_brl") < 5000, "large")
        .otherwise("whale")
    ) \
    .withColumn("is_high_risk", 
        (col("order_value_brl") > 5000).cast("boolean")
    ) \
    .withColumn("is_cross_state",
        (col("seller_state") != col("customer_state")).cast("boolean")
    )
print("Transformation defined")


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS shopsphere_catalog.retail.checkpoints;
SHOW VOLUMES IN shopsphere_catalog.retail;


In [0]:
# STEP C: Write Stream 1 — Raw enriched events to Delta
RAW_SINK_TABLE = "shopsphere_catalog.retail.order_events_enriched"
CHECKPOINT_RAW = "/Volumes/shopsphere_catalog/retail/checkpoints/raw_events"
# Create sink table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {RAW_SINK_TABLE}
    USING DELTA
    TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    AS SELECT * FROM {EVENT_SOURCE_TABLE} LIMIT 0
""")
# Align schema
target_cols = spark.table(RAW_SINK_TABLE).columns
order_stream_enriched = order_stream_enriched.select(target_cols)
# Write stream
raw_stream_query = (
    order_stream_enriched
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_RAW)
    .trigger(availableNow=True)
    .table(RAW_SINK_TABLE))
print(f"Raw stream query started. Query ID: {raw_stream_query.id}")
print(f"Status: {raw_stream_query.status['message']}")


In [0]:
# STEP D: Write Stream 2 - High-risk fraud alerts (filter)
from pyspark.sql.functions import current_timestamp, col
HIGH_RISK_TABLE = "shopsphere_catalog.risk.high_risk_orders"
CHECKPOINT_RISK = "/Volumes/shopsphere_catalog/retail/checkpoints/high_risk"
# Add missing column
order_stream_enriched = order_stream_enriched.withColumn(
    "ingest_timestamp", current_timestamp())
# Prepare stream
risk_df = (
    order_stream_enriched
    .filter(col("is_high_risk") == True)
    .select(
        "event_id", "order_id", "customer_id", "order_value_brl",
        "event_timestamp", "seller_state", "customer_state", "ingest_timestamp"))
# Create risk table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIGH_RISK_TABLE}
    USING DELTA
    COMMENT 'High-value orders flagged for fraud review'
    AS SELECT event_id, order_id, customer_id, order_value_brl, 
              event_timestamp, seller_state, customer_state
    FROM {EVENT_SOURCE_TABLE} LIMIT 0
""")
# ALIGN WITH TABLE
target_cols = spark.table(HIGH_RISK_TABLE).columns
risk_df = risk_df.select(target_cols)
# Write stream
risk_stream_query = (
    risk_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_RISK)
    .trigger(availableNow=True)
    .table(HIGH_RISK_TABLE))
print(f"Risk alert stream started. Query ID: {risk_stream_query.id}")


In [0]:
# Monitor all running streams
for stream in spark.streams.active:
    print(f"\n{'='*60}")
    print(f"Query: {stream.name or stream.id}")
    print(f"Status: {stream.status['message']}")
    print(f"Active: {stream.status['isDataAvailable']}")
    print(f"Trigger Active: {stream.status['isTriggerActive']}")
# Recent progress (last micro-batch metrics)
    if stream.lastProgress:
        prog = stream.lastProgress
        print(f"Input rows/sec: {prog.get('inputRowsPerSecond', 'N/A')}")
        print(f"Process rows/sec: {prog.get('processedRowsPerSecond', 'N/A')}")
        print(f"Batch ID: {prog.get('batchId', 'N/A')}")
        if 'sources' in prog:
            for src in prog['sources']:
                print(f"  Source lag: {src.get('numInputRows', 'N/A')} new rows")


In [0]:
# Generate more events to see the stream process them
from pyspark.sql.functions import col
import time
# Get target schema
target_cols = spark.table(EVENT_SOURCE_TABLE).columns
for batch_num in range(1, 6):
    new_events = generate_order_events(500, batch_id=batch_num)
    new_df = spark.createDataFrame(new_events, schema=schema)
    new_df = new_df.withColumnRenamed("order_value", "order_value_brl")
    new_df = new_df.select(target_cols)
    # Write safely
    new_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(EVENT_SOURCE_TABLE)
    print(f"Batch {batch_num} written: {new_df.count()} events")
    time.sleep(35)  # simulate streaming delay
# Check results
print(f"\nEnriched events: {spark.table('shopsphere_catalog.retail.order_events_enriched').count()}")
print(f"High-risk alerts: {spark.table('shopsphere_catalog.risk.high_risk_orders').count()}")


In [0]:
# Stop queries
raw_stream_query.stop()
risk_stream_query.stop()
# Verify stopped
print(f"Active streams after stop: {len(spark.streams.active)}")
CHECKPOINT_BASE = "/Volumes/shopsphere_catalog/retail/checkpoints"

# Verify checkpoint contents
checkpoint_contents = dbutils.fs.ls(CHECKPOINT_BASE)
for item in checkpoint_contents:
    print(f"Checkpoint dir: {item.name} ({item.size} bytes)")


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS shopsphere_catalog.retail.seller_reports;

In [0]:
LANDING_ZONE = "/Volumes/shopsphere_catalog/retail/seller_reports/landing"
ARCHIVE_ZONE = "/Volumes/shopsphere_catalog/retail/seller_reports/archive"
CHECKPOINT_AUTO = "/Volumes/shopsphere_catalog/retail/seller_reports/checkpoints"
OUTPUT_TABLE = "shopsphere_catalog.retail.seller_daily_reports"
# Create directory structure (now valid)
dbutils.fs.mkdirs(LANDING_ZONE)
dbutils.fs.mkdirs(ARCHIVE_ZONE)
dbutils.fs.mkdirs(CHECKPOINT_AUTO)
print(f"Landing zone: {LANDING_ZONE}")
print(f"Checkpoint: {CHECKPOINT_AUTO}")


In [0]:
# Step 2:  Generate Sample CSV Files
import random
import builtins
from datetime import datetime, timedelta
def create_seller_report_csv(seller_id, date_str, include_extra_col=False):
    """Create a realistic seller daily report CSV."""
    rows = ["seller_id,report_date,orders_count,gmv_brl,avg_rating,return_rate"]
    if include_extra_col:
        rows = ["seller_id,report_date,orders_count,gmv_brl,avg_rating,return_rate,nps_score"]
    for day_offset in range(7):
        order_count = random.randint(5, 500)
        gmv = builtins.round(order_count * random.uniform(80, 300), 2)
        rating = builtins.round(random.uniform(3.5, 5.0), 1)
        return_rate = builtins.round(random.uniform(0, 0.15), 3)
        row = f"{seller_id},{date_str},{order_count},{gmv},{rating},{return_rate}"
        if include_extra_col:
            nps = random.randint(50, 100)
            row += f",{nps}"
        rows.append(row)
    return "\n".join(rows)


In [0]:
# Write initial batch of seller reports (10 sellers, no extra column)
for i in range(1, 11):
    seller_id = f"SELLER_{i:04d}"
    content = create_seller_report_csv(seller_id, "2024-01-15")
    dbutils.fs.put(
        f"{LANDING_ZONE}/seller_report_{seller_id}_2024-01-15.csv",
        content,
        overwrite=True
    )
print(f"Written 10 seller report CSV files to landing zone")
print("Files:", [f.name for f in dbutils.fs.ls(LANDING_ZONE)])


In [0]:
# Configure Auto Loader Stream
from pyspark.sql.functions import col, input_file_name, current_timestamp
# Read with Auto Loader (cloudFiles format)
seller_stream = (
    spark.readStream
    .format("cloudFiles")          # This is Auto Loader
    .option("cloudFiles.format", "csv")           # Source file format
    .option("cloudFiles.inferColumnTypes", "true") # Infer proper types (not just strings)
    .option("header", "true")                      # CSV has header row
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Handle new columns
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_AUTO}/schema")  # Schema checkpoint
    .load(LANDING_ZONE)            # Monitor this path
    # Auto Loader adds metadata columns automatically:
    # _metadata.file_path, _metadata.file_modification_time, _metadata.file_size
)
print("Auto Loader stream configured")
seller_stream.printSchema()


In [0]:
# Add metadata and write to Delta
from pyspark.sql.functions import col, regexp_extract, current_timestamp
seller_stream_with_meta = (
    seller_stream
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("load_timestamp", current_timestamp())
    .withColumn(
        "extracted_seller_id",
        regexp_extract(col("_metadata.file_path"), r"seller_report_(SELLER_\d+)_", 1)
    )
)


In [0]:
# Write to Delta table
auto_loader_query = (
    seller_stream_with_meta
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AUTO)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(OUTPUT_TABLE))


In [0]:
# Wait for completion
auto_loader_query.awaitTermination()
print("Auto Loader batch completed!")
print(f"Rows loaded: {spark.table(OUTPUT_TABLE).count()}")
print(f"Files processed: {spark.table(OUTPUT_TABLE).select('source_file').distinct().count()}")


In [0]:
#  Test Incremental Loading - Add New Files
for i in range(11, 21):   # New sellers 11-20
    seller_id = f"SELLER_{i:04d}"
    content = create_seller_report_csv(seller_id, "2024-01-16")
    dbutils.fs.put(
        f"{LANDING_ZONE}/seller_report_{seller_id}_2024-01-16.csv",
        content, overwrite=True
    )
print("Added 10 more seller report files (new day)")


In [0]:
# Re-run Auto Loader - processes ONLY NEW files (11-20), not 1-10 again!
auto_loader_query2 = (
    seller_stream_with_meta
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AUTO)  # SAME checkpoint = knows which files already processed
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(OUTPUT_TABLE))
auto_loader_query2.awaitTermination()
print(f"Total rows after 2nd run: {spark.table(OUTPUT_TABLE).count()}")


In [0]:
# Test Schema Evolution
for i in range(21, 26):
    seller_id = f"SELLER_{i:04d}"
    #include_extra_col=True adds 'nps_score' column
    content = create_seller_report_csv(seller_id, "2024-01-17", include_extra_col=True)
    dbutils.fs.put(
        f"{LANDING_ZONE}/seller_report_{seller_id}_2024-01-17_v2.csv",
        content, overwrite=True
    )
print("Added 5 files with NEW COLUMN: nps_score")


In [0]:
# Run Auto Loader again with schema evolution
from pyspark.sql.functions import col, current_timestamp
auto_loader_query3 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_AUTO}/schema")
    .load(LANDING_ZONE)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("load_timestamp", current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AUTO)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(OUTPUT_TABLE)
)
auto_loader_query3.awaitTermination()


In [0]:
# Verify the new column exists
print("\nTable schema after evolution:")
spark.table(OUTPUT_TABLE).printSchema()
# The 'nps_score' column should appear!
# Old rows will have NULL for nps_score (backward compatible)
print(f"\nRows with nps_score: {spark.table(OUTPUT_TABLE).filter('nps_score IS NOT NULL').count()}")
print(f"Rows without nps_score: {spark.table(OUTPUT_TABLE).filter('nps_score IS NULL').count()}")
